In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hugoaslm/fm-change-detection-benchmark.git"
BRANCH = "main"
REPO_DIR = Path("/content/fmcd-selection")
ZIP_PATH = Path("/content/drive/MyDrive/datasets/LEVIR-CD256.zip")
EXTRACT_DIR = Path("/content/datasets/levir-cd-256")
SELECTION_DRIVE_DIR = Path("/content/drive/MyDrive/fm-change-detection/selection")
FINAL_DRIVE_DIR = Path("/content/drive/MyDrive/fm-change-detection/final-test")


RUN_SELECTION = True
RUN_FINAL_TEST = False
RUN_FRONTIER = False

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git clone; rename or remove it.")
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
import importlib
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", "-e", ".[dev]"], check=True
)


sys.path.insert(0, str(REPO_DIR / "src"))
for module_name in list(sys.modules):
    if module_name == "fm_change_detection" or module_name.startswith("fm_change_detection."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import fm_change_detection

source_path = Path(fm_change_detection.__file__).resolve()
print("Imported package:", source_path)
assert str(source_path).startswith(str(REPO_DIR)), "Python imported a stale package checkout."

help_result = subprocess.run(
    [sys.executable, "-m", "fm_change_detection.cli", "--help"],
    check=True,
    capture_output=True,
    text=True,
)
print(help_result.stdout)
assert "select" in help_result.stdout, "The checked-out repository does not contain fmcd select."

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Switch this Colab runtime to a GPU before selection."

subprocess.run([sys.executable, "-m", "fm_change_detection.cli", "smoke"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
import shutil

from google.colab import drive

drive.mount("/content/drive")
assert ZIP_PATH.is_file(), f"Dataset ZIP not found: {ZIP_PATH}"


def find_dataset_root(search_root: Path):
    paths = [search_root, *search_root.rglob("*")]
    return next(
        (
            path
            for path in paths
            if path.is_dir() and all((path / name).is_dir() for name in ("A", "B", "label", "list"))
        ),
        None,
    )


DATA_ROOT = find_dataset_root(EXTRACT_DIR) if EXTRACT_DIR.exists() else None
if DATA_ROOT is None:
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True)
    print("Extracting LEVIR-CD256.zip...")
    shutil.unpack_archive(str(ZIP_PATH), str(EXTRACT_DIR))
    DATA_ROOT = find_dataset_root(EXTRACT_DIR)

assert DATA_ROOT is not None, "Could not locate A/, B/, label/, and list/ after extraction."
print("Dataset root:", DATA_ROOT)

subprocess.run(
    [
        sys.executable,
        "-m",
        "fm_change_detection.cli",
        "validate-data",
        "--root",
        str(DATA_ROOT),
    ],
    check=True,
)

In [ ]:
SELECTION_DIR = REPO_DIR / "outputs" / "selection"
selection_artifact_names = (
    "candidates.csv",
    "selection.json",
    "selection.md",
    "final_selected.yaml",
)

if RUN_SELECTION:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "fm_change_detection.cli",
            "select",
            "--config",
            "configs/selection.yaml",
            "--data-root",
            str(DATA_ROOT),
            "--device",
            "cuda",
        ],
        check=True,
    )
else:
    SELECTION_DIR.mkdir(parents=True, exist_ok=True)
    missing = [
        name for name in selection_artifact_names if not (SELECTION_DRIVE_DIR / name).is_file()
    ]
    assert not missing, (
        f"Missing saved selection artifacts in {SELECTION_DRIVE_DIR}: {missing}. "
        "Upload them there or set RUN_SELECTION=True."
    )
    for name in selection_artifact_names:
        shutil.copy2(SELECTION_DRIVE_DIR / name, SELECTION_DIR / name)
    print("Restored the inspected validation selection from:", SELECTION_DRIVE_DIR)

In [ ]:
from IPython.display import Markdown, display

required_artifacts = [SELECTION_DIR / name for name in selection_artifact_names]
assert all(path.is_file() for path in required_artifacts), "Selection artifacts are incomplete."

display(Markdown((SELECTION_DIR / "selection.md").read_text()))
print("\n--- Frozen candidate configuration ---\n")
print((SELECTION_DIR / "final_selected.yaml").read_text())

In [ ]:
SELECTION_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

for source in required_artifacts:
    shutil.copy2(source, SELECTION_DRIVE_DIR / source.name)

archive_path = shutil.make_archive("/content/validation_selection_artifacts", "zip", SELECTION_DIR)
print("Saved persistent artifacts to:", SELECTION_DRIVE_DIR)
print("Created downloadable archive:", archive_path)

In [ ]:
from google.colab.files import download

download("/content/validation_selection_artifacts.zip")

In [ ]:
import json

import yaml

selection_record = json.loads((SELECTION_DIR / "selection.json").read_text())
final_config = yaml.safe_load((SELECTION_DIR / "final_selected.yaml").read_text())

assert selection_record["accessed_splits"] == ["train", "val"]
assert final_config["runtime"]["max_test_samples"] is None
assert final_config["bootstrap"]["num_resamples"] == 0

recorded_winners = {
    (row["encoder"], row["layer"], row["score"]) for row in selection_record["selected"]
}
configured_winners = {
    (encoder["name"], encoder["layers"][0], encoder["scores"][0])
    for encoder in final_config["encoders"]
}
assert configured_winners == recorded_winners, "Frozen YAML does not match selection.json."
assert len(configured_winners) == 4, "Expected one frozen candidate per representation family."

print("Frozen candidates verified:")
for candidate in sorted(configured_winners):
    print(" -", candidate)
print("Selection commit:", selection_record["git_commit"])

In [ ]:
if not RUN_FINAL_TEST:
    print("Final test is disabled. Set RUN_FINAL_TEST=True only after inspecting the selection.")
else:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "fm_change_detection.cli",
            "benchmark",
            "--config",
            str(SELECTION_DIR / "final_selected.yaml"),
            "--data-root",
            str(DATA_ROOT),
            "--device",
            "cuda",
        ],
        check=True,
    )

In [ ]:
if RUN_FINAL_TEST:
    import pandas as pd

    FINAL_RESULTS_DIR = REPO_DIR / "outputs" / "final_results"
    FINAL_REPORT = REPO_DIR / "reports" / "benchmark.md"
    result_files = sorted(FINAL_RESULTS_DIR.glob("*.json"))
    assert len(result_files) == 8, f"Expected 8 result JSON files, found {len(result_files)}."

    test_count = sum(
        1 for line in (DATA_ROOT / "list" / "test.txt").read_text().splitlines() if line.strip()
    )
    records = [json.loads(path.read_text()) for path in result_files]
    assert all(record["num_images"] == test_count for record in records)
    assert all(record["max_test_samples"] is None for record in records)
    assert len({record["manifest_hash"] for record in records}) == 1

    summary = pd.read_csv(FINAL_RESULTS_DIR / "summary.csv")
    display(
        summary[
            [
                "encoder",
                "layer",
                "score",
                "threshold_method",
                "average_precision",
                "auroc",
                "f1",
                "iou",
            ]
        ].sort_values(["threshold_method", "average_precision"], ascending=[True, False])
    )
    display(Markdown(FINAL_REPORT.read_text()))
    print(f"Verified complete test evaluation on {test_count} image pairs.")

In [ ]:
if RUN_FINAL_TEST:
    FINAL_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    drive_results = FINAL_DRIVE_DIR / "results"
    if drive_results.exists():
        shutil.rmtree(drive_results)
    shutil.copytree(FINAL_RESULTS_DIR, drive_results)
    shutil.copy2(FINAL_REPORT, FINAL_DRIVE_DIR / "benchmark.md")
    shutil.copy2(SELECTION_DIR / "selection.json", FINAL_DRIVE_DIR / "selection.json")
    shutil.copy2(SELECTION_DIR / "final_selected.yaml", FINAL_DRIVE_DIR / "final_selected.yaml")

    archive_root = Path("/content/final_test_artifacts")
    if archive_root.exists():
        shutil.rmtree(archive_root)
    shutil.copytree(FINAL_DRIVE_DIR, archive_root)
    final_archive = shutil.make_archive("/content/final_test_artifacts", "zip", archive_root)
    print("Saved persistent final artifacts to:", FINAL_DRIVE_DIR)
    print("Created downloadable archive:", final_archive)

In [ ]:
if RUN_FINAL_TEST:
    download("/content/final_test_artifacts.zip")

In [ ]:
if not RUN_FRONTIER:
    print("Frontier is disabled. Set RUN_FRONTIER=True after the final test is frozen.")
else:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "fm_change_detection.cli",
            "frontier",
            "--config",
            "configs/detectability.yaml",
            "--data-root",
            str(DATA_ROOT),
            "--device",
            "cuda",
        ],
        check=True,
    )

In [ ]:
if RUN_FRONTIER:
    from IPython.display import Image

    FRONTIER_REPORT = REPO_DIR / "reports" / "frontier.md"
    assert FRONTIER_REPORT.is_file(), "Frontier report missing after the run."
    display(Markdown(FRONTIER_REPORT.read_text()))

    fig_dir = REPO_DIR / "reports" / "figures"
    for fig in sorted(fig_dir.glob("frontier_*.png")):
        display(Image(filename=str(fig), width=720))

    FRONTIER_DRIVE_DIR = Path("/content/drive/MyDrive/fm-change-detection/frontier")
    FRONTIER_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    drive_results = FRONTIER_DRIVE_DIR / "results"
    if drive_results.exists():
        shutil.rmtree(drive_results)
    shutil.copytree(REPO_DIR / "outputs" / "results_frontier", drive_results)
    shutil.copy2(FRONTIER_REPORT, FRONTIER_DRIVE_DIR / "frontier.md")
    drive_figs = FRONTIER_DRIVE_DIR / "figures"
    if drive_figs.exists():
        shutil.rmtree(drive_figs)
    shutil.copytree(fig_dir, drive_figs)
    print("Saved persistent frontier artifacts to:", FRONTIER_DRIVE_DIR)